In [1]:
import torch
import torch.nn as nn
import numpy as np
import math
from torch.utils.data import Dataset, DataLoader

In [ ]:
class RNN:
    def __init__(self, input_size, hidden_size, output_size):
        self.hs = hidden_size
        self.is_ = input_size
        self.os = output_size

        self.Wxh = torch.randn(hidden_size, input_size) * 0.01
        self.Whh = torch.randn(hidden_size, hidden_size) * 0.01
        self.Wxy = torch.randn(output_size, hidden_size) * 0.01

        self.bh = torch.zeros(hidden_size, 1)
        self.by = torch.zeros(output_size, 1)

        self.h_history = []
        self.y_history = []

    def forward(self, X):
        h_prev = torch.zeros(self.hs, 1)
        self.X_history = X
        self.h_history = [h_prev]
        self.y_history = []

        for t in range(len(X)):
            xt = X[t]
            zt = torch.matmul(self.Wxh, xt) + torch.matmul(self.Whh, h_prev) + self.bh
            ht = torch.tanh(zt)
            yt = torch.matmul(self.Wxy, ht) + self.by

            self.h_history.append(ht)
            self.y_history.append(yt)
            h_prev = ht 
        return self.y_history 

    def backward(self, dY):
        dWxh = torch.zeros_like(self.Wxh)
        dWhh = torch.zeros_like(self.Whh)
        dWxy = torch.zeros_like(self.Wxy)
        dbh = torch.zeros_like(self.bh)
        dby = torch.zeros_like(self.by)

        dh_next = torch.zeros(self.hs, 1)
        seq_len = len(self.X_history)

        for t in reversed(range(seq_len)):
            xt = self.X_history[t]
            ht = self.h_history[t + 1]
            h_prev = self.h_history[t] 

            dyt = dY[t]
            dWxy += torch.matmul(dyt, ht.t())
            dby += dyt

            dht = torch.matmul(self.Wxy.t(), dyt) + dh_next
            dzt = dht * (1 - ht ** 2)

            dWxh += torch.matmul(dzt, xt.t())
            dWhh += torch.matmul(dzt, h_prev.t())
            dbh += dzt

            dh_next = torch.matmul(self.Whh.t(), dzt)

        return dWxh, dWhh, dWxy, dbh, dby

    def update_parameters(self, dWxh, dWhh, dWxy, dbh, dby, learning_rate=0.01):
        self.Wxh -= learning_rate * dWxh
        self.Whh -= learning_rate * dWhh
        self.Wxy -= learning_rate * dWxy
        self.bh -= learning_rate * dbh
        self.by -= learning_rate * dby
        return

    def train(self, X, Y, learning_rate=0.01):
        y_pred = self.forward(X)
        dy_history = [y_pred[t] - Y[t] for t in range(len(Y))]
        dWxh, dWhh, dWxy, dbh, dby = self.backward(dy_history)
        self.update_parameters(dWxh, dWhh, dWxy, dbh, dby, learning_rate)
        return y_pred

In [3]:
class SineWaveDataset(Dataset):
    def __init__(self, seq_length, num_samples):
        t = torch.linspace(0, 20 * math.pi, num_samples + seq_length)
        self.data = torch.sin(t)
        self.seq_len = seq_length

    def __len__(self):
        return len(self.data) - self.seq_len

    def __getitem__(self, idx):
        X_seq = self.data[idx : idx + self.seq_len].view(self.seq_len, 1, 1)
        Y_seq = self.data[idx + 1 : idx + self.seq_len + 1].view(self.seq_len, 1, 1)

        return X_seq, Y_seq


In [4]:
dataset = SineWaveDataset(seq_length=10, num_samples=1000)
dataset_loader = DataLoader(dataset, batch_size=1, shuffle=True)
model = RNN(input_size=1, hidden_size=10, output_size=1)
epochs = 20
learning_rate = 0.05

print("Training the RNN model...")
for epoch in range(epochs):
    epoch_loss = 0.0
    for batch_idx, (X,Y) in enumerate(dataset_loader):
        x_seq = X[0]
        y_seq = Y[0]

        y_pred = model.forward(x_seq)
        dy_history = []
        seq_loss = 0.0
        for t in range(len(y_seq)):
            error = y_pred[t] - y_seq[t]
            dy_history.append(error)
            seq_loss += (error.item() ** 2)
        epoch_loss += seq_loss / len(y_seq)

        dWxh, dWhh, dWxy, dbh, dby = model.backward(dy_history)
        model.update_parameters(dWxh, dWhh, dWxy, dbh, dby, learning_rate)

    print(f"Epoch {epoch + 1}/{epochs}, Loss: {epoch_loss / len(dataset_loader)}")


Training the RNN model...
Epoch 1/20, Loss: 0.017777363053273966
Epoch 2/20, Loss: 0.005260031549578981
Epoch 3/20, Loss: 0.005129283403249319
Epoch 4/20, Loss: 0.004764699454446569
Epoch 5/20, Loss: 0.004605555189032106
Epoch 6/20, Loss: 0.0045291159355260015
Epoch 7/20, Loss: 0.004162372001411265
Epoch 8/20, Loss: 0.004029596407356727
Epoch 9/20, Loss: 0.0033272933853615494
Epoch 10/20, Loss: 0.0032651975336876735
Epoch 11/20, Loss: 0.0028510620749870675
Epoch 12/20, Loss: 0.002653549894255785
Epoch 13/20, Loss: 0.002430632533317223
Epoch 14/20, Loss: 0.002322964971766305
Epoch 15/20, Loss: 0.0022051342205816234
Epoch 16/20, Loss: 0.002255314084384922
Epoch 17/20, Loss: 0.0021540710867736673
Epoch 18/20, Loss: 0.002158459523505928
Epoch 19/20, Loss: 0.0020169543129352095
Epoch 20/20, Loss: 0.0020424856765601327


In [8]:
import torch.optim as optim
import os

In [6]:
class AutoGradRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.hs = hidden_size

        self.wxh = nn.Parameter(torch.randn(hidden_size, input_size) * 0.01)
        self.whh = nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.01)
        self.wxy = nn.Parameter(torch.randn(output_size, hidden_size) * 0.01)

        self.bh = nn.Parameter(torch.zeros(hidden_size, 1))
        self.by = nn.Parameter(torch.zeros(output_size, 1))

    def forward(self, X):
        h_prev = torch.zeros(self.hs, 1)
        y_history = []

        for t in range(len(X)):
            xt = X[t]
            zt = torch.matmul(self.wxh, xt) + torch.matmul(self.whh, h_prev) + self.bh
            ht = torch.tanh(zt)
            yt = torch.matmul(self.wxy, ht) + self.by

            y_history.append(yt)
            h_prev = ht
        return torch.stack(y_history)
    

In [9]:
model = AutoGradRNN(input_size=1, hidden_size=10, output_size=1)
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.05)


for epoch in range(epochs):
    epoch_loss = 0.0
    best_loss = float('inf')
    for batch_idx, (X,Y) in enumerate(dataset_loader):
        x_seq = X[0]
        y_seq = Y[0]

        optimizer.zero_grad()
        y_pred = model(x_seq)
        loss = criterion(y_pred, y_seq)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        if loss.item() < best_loss:
            best_loss = loss.item()
            checkpoint = {
                'epoch' : epoch + 1,
                'model_state_dict' : model.state_dict(),
                'optimizer_state_dict' : optimizer.state_dict(),
                'loss' : best_loss
            }
            if os.path.exists('checkpoints') == False:
                os.makedirs('checkpoints')
            torch.save(checkpoint, f'checkpoints/checkpoint_epoch_{epoch + 1}.pt')


    print(f"Epoch {epoch + 1}/{epochs}, Loss: {epoch_loss / len(dataset_loader)}")

Epoch 1/20, Loss: 0.04221178790269596
Epoch 2/20, Loss: 0.0022458374702609943
Epoch 3/20, Loss: 0.002214144557929103
Epoch 4/20, Loss: 0.0022517724719527906
Epoch 5/20, Loss: 0.0022618929655691318
Epoch 6/20, Loss: 0.0022114918040842895
Epoch 7/20, Loss: 0.002174531251548615
Epoch 8/20, Loss: 0.0022120251792694034
Epoch 9/20, Loss: 0.002198252090787719
Epoch 10/20, Loss: 0.00213631198434814
Epoch 11/20, Loss: 0.0021756806376815804
Epoch 12/20, Loss: 0.0021120464716441346
Epoch 13/20, Loss: 0.002172587265726179
Epoch 14/20, Loss: 0.0020972340323769456
Epoch 15/20, Loss: 0.0021275751962002687
Epoch 16/20, Loss: 0.002093735107122484
Epoch 17/20, Loss: 0.0020999708473345893
Epoch 18/20, Loss: 0.002093795139935537
Epoch 19/20, Loss: 0.002101902358048392
Epoch 20/20, Loss: 0.002054686595283783


In [10]:
# load the best model checkpoint
checkpoint_files = [f for f in os.listdir('checkpoints') if f.startswith('checkpoint_epoch_')]
best_checkpoint = None
best_loss = float('inf')
for checkpoint_file in checkpoint_files:
    checkpoint = torch.load(os.path.join('checkpoints', checkpoint_file))
    if checkpoint['loss'] < best_loss:
        best_loss = checkpoint['loss']
        best_checkpoint = checkpoint

if best_checkpoint is not None:
    model.load_state_dict(best_checkpoint['model_state_dict'])
    optimizer.load_state_dict(best_checkpoint['optimizer_state_dict'])
    print(f"Loaded best model from epoch {best_checkpoint['epoch']} with loss {best_checkpoint['loss']}")
else:
    print("No checkpoint found.")

Loaded best model from epoch 1 with loss 4.157319835940143e-06


In [11]:
class ProductionRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, X):
        batch_size = X.size(0)
        h0 = torch.zeros(1, batch_size, self.hidden_size)
        out, _ = self.rnn(X, h0)
        out = self.fc(out)
        return out